# FinSight AI — Google Colab Setup

Global ticker-first development/smoke test companion for the Streamlit application.
Try US, European, Japanese and Indian examples such as `AAPL`, `TSLA`, `SAP.DE`, `7203.T`, and `RELIANCE.NS`.


In [ ]:
!pip -q install -r requirements.txt


In [ ]:
import os
# Optional: set your Groq key only for this Colab session.
os.environ['GROQ_API_KEY'] = 'PASTE_YOUR_KEY_HERE'
os.environ['GROQ_MODEL'] = 'openai/gpt-oss-20b'


In [ ]:
from services.market_data import get_price_history, get_fundamentals, normalize_symbol
from analytics.technical import build_analysis

for query in ['AAPL', 'TSLA', 'SAP.DE', 'RELIANCE.NS']:
    symbol = normalize_symbol(query)
    history = get_price_history(symbol, period='6mo', interval='1d')
    analysis = build_analysis(history)
    print(query, '→', symbol, '| rows:', len(history), '| signal:', analysis.attrs.get('signal'))


In [ ]:
from services.news import get_news
symbol='AAPL'
news=get_news(symbol)
for item in news[:5]:
    print(item['label'], item['title'])


In [ ]:
from services.llm import generate_ai_report
from services.market_data import get_market_snapshot

snapshot=get_market_snapshot(symbol, history)
fundamentals=get_fundamentals(symbol)
report=generate_ai_report(
    symbol, snapshot, fundamentals,
    {'rsi14':float(analysis['RSI14'].iloc[-1]),
     'sma20':float(analysis['SMA20'].iloc[-1]),
     'sma50':float(analysis['SMA50'].iloc[-1]),
     'volatility':float(analysis.attrs['annualized_volatility']),
     'signal':analysis.attrs['signal']}, news[:8])
print(report)


In [ ]:
!pytest -q
